# VOICE_CLONE_TTS — Nhân bản giọng nói (Text-to-Speech) — Bản nguồn mở
## Cách dùng nhanh: bấm "Chạy tất cả" (Run all), đợi khoảng 5 phút cài đặt là dùng được.

VUI LÒNG ĐỌC HẾT HƯỚNG DẪN BÊN DƯỚI TRƯỚC KHI SỬ DỤNG.

---

## GIỚI THIỆU
* Bản xây dựng sạch từ mã nguồn mở **F5-TTS** (https://github.com/SWivid/F5-TTS), minh bạch, không dùng app mã hóa.
* Chức năng: **nhân bản giọng nói** — tải lên 1 file giọng mẫu (5–12 giây), nhập văn bản bất kỳ, máy sẽ đọc bằng đúng giọng đó.
* Hỗ trợ **KHÔNG GIỚI HẠN KÍ TỰ** (tự động chia nhỏ văn bản dài).
* **Đã tối ưu chống thiếu chữ / vấp**: văn bản được chia theo câu với độ dài cố định, mỗi đoạn được kết thúc đầy đủ dấu câu trước khi ghép nối.
* Model mặc định: **F5-TTS v1 Base (Anh + Trung Quốc)**. Giao diện có sẵn hơn **10 model ngôn ngữ**: tiếng Việt, Anh, Trung, Nhật, Pháp, Đức, Ý, Tây Ban Nha, Nga, Phần Lan, Latvia, Hindi...

## 👉 HƯỚNG DẪN:
1. Trên trình đơn Colab: **Runtime -> Change runtime type -> T4 GPU** (cực kì quan trọng).
2. **Bấm "Chạy tất cả"** để cài đặt và khởi động.
3. Lần chạy đầu tiên sẽ tải model (~1.3 GB) và mất vài phút.
4. Sau khi có link giao diện (dòng cuối cùng của TASK 6), mở trong tab mới.
5. Tải lên **file giọng mẫu** (WAV/MP3 sạch, không nhạc nền, 5–12 giây), nhập văn bản, bấm **Synthesize**.
6. Chọn **model theo ngôn ngữ** cần đọc (vd. `Tiếng Việt`, `Tiếng Nhật`, `Tiếng Pháp`...). Lần đầu chọn model nào máy sẽ tải model đó (~1.3 GB).
7. **Lưu giọng để dùng lại**: điền Tên giọng rồi bấm `Lưu giọng mẫu này` — lần sau chỉ cần chọn tên trong danh sách `Giọng đã lưu` là có lại đúng file giọng + nội dung cũ.
8. (Tùy chọn) Chạy cell **TASK 6 — GẮN GOOGLE DRIVE** để giọng đã lưu được giữ lâu dài giữa các phiên Colab.

## NẾU VẪN CÒN THIẾU CHỮ / VẤP:
* Giảm thanh trượt **"Độ dài đoạn (max_chars)"** xuống 60–80.
* Nhập chính xác **"Nội dung trong file giọng mẫu"** (không để trống) để máy tính đúng độ dài.
* Dùng file giọng mẫu ngắn, rõ (5–10 giây), không nhạc nền.
* Tăng **nfe_step** lên 48–64 nếu bị rè hoặc vấp.

## NẾU KẾT QUẢ NÓI THỪA / LẶP LẠI 1 ĐOẠN VĂN BẢN:
* Nguyên nhân chính: model nhồi "Nội dung trong file giọng mẫu" (ref_text) vào chung chuỗi sinh audio, khi độ dài giọng mẫu bị sai/lệch thì **đuôi của ref_text bị đọc thêm ra**. Đoạn thừa đó chính là văn bản bạn đã nhập cho giọng mẫu lúc trước.
* Máy sẽ **cảnh báo** nếu giọng mẫu dài hơn 12 giây (lúc đó F5-TTS tự cắt cụt giọng mẫu nhưng giữ nguyên nội dung — dễ gây thừa/lặp). Nếu vẫn còn thừa:
* Dùng file giọng mẫu **dưới 12 giây**, cắt gọn (đầu/cuối có chút lặng), không nhạc nền.
* Nếu không chắc "Nội dung trong file giọng mẫu" khớp 100%, hãy **để trống** để máy tự nhận dạng đúng độ dài.
* Tăng **nfe_step** lên 48–64, hoặc tăng nhẹ **cfg_strength** lên 2.5–3.0 để giảm ảo giác lặp chữ.

## ⚠️ LƯU Ý:
* Chỉ dùng cho mục đích cá nhân / học tập. **Không** nhân bản giọng nói của người khác khi chưa được phép.
* Model F5-TTS dùng giấy phép **CC-BY-NC** (phi thương mại).


In [ ]:
# @title TASK 1 — KIỂM TRA MÔI TRƯỜNG (GPU / FFMPEG)import base64 as _b64exec(_b64.b64decode("IyBAdGl0bGUgVEFTSyAxIOKAlCBLSeG7gk0gVFJBIE3DlEkgVFLGr+G7nE5HIChHUFUgLyBGRk1QRUcpCmltcG9ydCBzdWJwcm9jZXNzLCBzeXMKCiMgMSkgS2nhu4NtIHRyYSBHUFUKaW1wb3J0IHRvcmNoCnByaW50KCJDVURBIGF2YWlsYWJsZToiLCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpKQppZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgcHJpbnQoIkdQVToiLCB0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKSkKZWxzZToKICAgIHByaW50KCJD4bqiTkggQsOBTzogS2jDtG5nIGPDsyBHUFUuIFbDoG8gUnVudGltZSAtPiBDaGFuZ2UgcnVudGltZSB0eXBlIC0+IGNo4buNbiBUNCBHUFUgcuG7k2kgY2jhuqF5IGzhuqFpLiIpCgojIDIpIEPDoGkgZmZtcGVnIG7hur91IGNoxrBhIGPDsyAoY+G6p24gxJHhu4MgxJHhu41jL3h14bqldCDDom0gdGhhbmgpCnRyeToKICAgIHN1YnByb2Nlc3MucnVuKFsiZmZtcGVnIiwgIi12ZXJzaW9uIl0sIHN0ZG91dD1zdWJwcm9jZXNzLkRFVk5VTEwsIHN0ZGVycj1zdWJwcm9jZXNzLkRFVk5VTEwsIGNoZWNrPVRydWUpCiAgICBwcmludCgiZmZtcGVnOiBPSyIpCmV4Y2VwdCBFeGNlcHRpb246CiAgICBwcmludCgiZmZtcGVnOiDEkWFuZyBjw6BpIMSR4bq3dC4uLiIpCiAgICBzdWJwcm9jZXNzLnJ1bihbImFwdC1nZXQiLCAidXBkYXRlIiwgIi1xcSJdLCBjaGVjaz1GYWxzZSkKICAgIHN1YnByb2Nlc3MucnVuKFsiYXB0LWdldCIsICJpbnN0YWxsIiwgIi15IiwgIi1xcSIsICJmZm1wZWciXSwgY2hlY2s9RmFsc2UpCiAgICBwcmludCgiZmZtcGVnOiB4b25nIikK").decode("utf-8"))

In [ ]:
# @title TASK 2 — CÀI ĐẶT F5-TTSimport base64 as _b64exec(_b64.b64decode("IyBAdGl0bGUgVEFTSyAyIOKAlCBDw4BJIMSQ4bq2VCBGNS1UVFMKIyBUcsOqbiBDb2xhYiwgdG9yY2gvdG9yY2hhdWRpbyDEkcOjIGPDsyBz4bq1bi4gcGlwIGluc3RhbGwgZjUtdHRzIHPhur0gYuG7lSBzdW5nIGPDoWMgdGjGsCB2aeG7h24gY8OybiB0aGnhur91LgppbXBvcnQgc3VicHJvY2Vzcywgc3lzCnRyeToKICAgIGltcG9ydCBmNV90dHMgICMgbm9xYTogRjQwMQogICAgcHJpbnQoIuKchSBGNS1UVFMgxJHDoyDEkcaw4bujYyBjw6BpIHPhurVuIOKAlCBi4buPIHF1YSBjw6BpIMSR4bq3dCwgc2FuZyBixrDhu5tjIHRp4bq/cCB0aGVvLiIpCmV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgIHN1YnByb2Nlc3MucnVuKFtzeXMuZXhlY3V0YWJsZSwgIi1tIiwgInBpcCIsICJpbnN0YWxsIiwgIi1xIiwgImY1LXR0cyJdLCBjaGVjaz1UcnVlKQogICAgcHJpbnQoIsSQw6MgY8OgaSB4b25nIEY1LVRUUy4iKQo=").decode("utf-8"))

In [ ]:
# @title TASK 3 — TẢI & NẠP MODEL (nhiều ngôn ngữ)import base64 as _b64exec(_b64.b64decode("IyBAdGl0bGUgVEFTSyAzIOKAlCBU4bqiSSAmIE7huqBQIE1PREVMIChuaGnhu4F1IG5nw7RuIG5n4buvKQppbXBvcnQgb3MKZnJvbSBmNV90dHMuYXBpIGltcG9ydCBGNVRUUwpmcm9tIGh1Z2dpbmdmYWNlX2h1YiBpbXBvcnQgaGZfaHViX2Rvd25sb2FkCgojIERhbmggc8OhY2ggbW9kZWwgbmfDtG4gbmfhu68gY8OzIHPhurVuLiBNb2RlbCDEkcaw4bujYyB04bqjaSAibMaw4budaSIgKGNo4buJIHThuqNpIGtoaSBjaOG7jW4gdHJvbmcgZ2lhbyBkaeG7h24pLgpNT0RFTFMgPSB7CiAgICAi8J+Hu/Cfh7MgVGnhur9uZyBWaeG7h3QgKEY1LVRUUy1WaWV0bmFtZXNlKSI6IHsibW9kZWwiOiAiRjVUVFNfQmFzZSIsICJyZXBvX2lkIjogInRvYW5kZXYvRjUtVFRTLVZpZXRuYW1lc2UiLCAic3ViZm9sZGVyIjogTm9uZSwgImNrcHRfbmFtZSI6ICJtb2RlbF9sYXRlc3Quc2FmZXRlbnNvcnMiLCAidm9jYWJfbmFtZSI6ICJ2b2NhYi50eHQifSwKICAgICLwn4es8J+Hp/Cfh6jwn4ezIEY1LVRUUyB2MSBCYXNlIChBbmggKyBUcnVuZykiOiB7Im1vZGVsIjogIkY1VFRTX3YxX0Jhc2UiLCAicmVwb19pZCI6IE5vbmUsICJzdWJmb2xkZXIiOiBOb25lLCAiY2twdF9uYW1lIjogTm9uZSwgInZvY2FiX25hbWUiOiBOb25lfSwKICAgICLwn4es8J+Hp/Cfh6jwn4ezIEY1LVRUUyBCYXNlIChBbmggKyBUcnVuZykiOiB7Im1vZGVsIjogIkY1VFRTX0Jhc2UiLCAicmVwb19pZCI6IE5vbmUsICJzdWJmb2xkZXIiOiBOb25lLCAiY2twdF9uYW1lIjogTm9uZSwgInZvY2FiX25hbWUiOiBOb25lfSwKICAgICLwn4ev8J+HtSBUaeG6v25nIE5o4bqtdCAoSm1pY2EvRjVUVFMpIjogeyJtb2RlbCI6ICJGNVRUU19CYXNlIiwgInJlcG9faWQiOiAiSm1pY2EvRjVUVFMiLCAic3ViZm9sZGVyIjogIkpBXzIxOTk5MTIwIiwgImNrcHRfbmFtZSI6ICJtb2RlbF8yMTk5OTEyMC5wdCIsICJ2b2NhYl9uYW1lIjogInZvY2FiX2phcGFuZXNlLnR4dCJ9LAogICAgIvCfh6vwn4e3IFRp4bq/bmcgUGjDoXAgKFJBU1BJQVVESU8vRjUtRnJlbmNoKSI6IHsibW9kZWwiOiAiRjVUVFNfQmFzZSIsICJyZXBvX2lkIjogIlJBU1BJQVVESU8vRjUtRnJlbmNoLU1peGVkU3BlYWtlcnMtcmVkdWNlZCIsICJzdWJmb2xkZXIiOiBOb25lLCAiY2twdF9uYW1lIjogIm1vZGVsX2xhc3RfcmVkdWNlZC5wdCIsICJ2b2NhYl9uYW1lIjogInZvY2FiLnR4dCJ9LAogICAgIvCfh6nwn4eqIFRp4bq/bmcgxJDhu6ljIChodm9zcy10ZWNoZmFrL0Y1LVRUUy1HZXJtYW4pIjogeyJtb2RlbCI6ICJGNVRUU19CYXNlIiwgInJlcG9faWQiOiAiaHZvc3MtdGVjaGZhay9GNS1UVFMtR2VybWFuIiwgInN1YmZvbGRlciI6IE5vbmUsICJja3B0X25hbWUiOiAibW9kZWxfZjV0dHNfZ2VybWFuLnB0IiwgInZvY2FiX25hbWUiOiAidm9jYWIudHh0In0sCiAgICAi8J+HrvCfh7kgVGnhur9uZyDDnSAoYWxpZW43OS9GNS1UVFMtaXRhbGlhbikiOiB7Im1vZGVsIjogIkY1VFRTX0Jhc2UiLCAicmVwb19pZCI6ICJhbGllbjc5L0Y1LVRUUy1pdGFsaWFuIiwgInN1YmZvbGRlciI6IE5vbmUsICJja3B0X25hbWUiOiAibW9kZWxfMTU5NjAwLnNhZmV0ZW5zb3JzIiwgInZvY2FiX25hbWUiOiAidm9jYWIudHh0In0sCiAgICAi8J+HqvCfh7ggVGnhur9uZyBUw6J5IEJhbiBOaGEgKGpwZ2FsbGVnb2FyL0Y1LVNwYW5pc2gpIjogeyJtb2RlbCI6ICJGNVRUU19CYXNlIiwgInJlcG9faWQiOiAianBnYWxsZWdvYXIvRjUtU3BhbmlzaCIsICJzdWJmb2xkZXIiOiBOb25lLCAiY2twdF9uYW1lIjogIm1vZGVsX2xhc3QucHQiLCAidm9jYWJfbmFtZSI6ICJ2b2NhYi50eHQifSwKICAgICLwn4e38J+HuiBUaeG6v25nIE5nYSAoaG90c3RvbmUyMjgvRjUtVFRTLVJ1c3NpYW4pIjogeyJtb2RlbCI6ICJGNVRUU19CYXNlIiwgInJlcG9faWQiOiAiaG90c3RvbmUyMjgvRjUtVFRTLVJ1c3NpYW4iLCAic3ViZm9sZGVyIjogTm9uZSwgImNrcHRfbmFtZSI6ICJtb2RlbF9sYXN0LnNhZmV0ZW5zb3JzIiwgInZvY2FiX25hbWUiOiAidm9jYWIudHh0In0sCiAgICAi8J+Hq/Cfh64gVGnhur9uZyBQaOG6p24gTGFuIChBc21vS29za2luZW4vRjUtVFRTLUZpbm5pc2gpIjogeyJtb2RlbCI6ICJGNVRUU19CYXNlIiwgInJlcG9faWQiOiAiQXNtb0tvc2tpbmVuL0Y1LVRUU19GaW5uaXNoX01vZGVsIiwgInN1YmZvbGRlciI6IE5vbmUsICJja3B0X25hbWUiOiAibW9kZWxfY29tbW9uX3ZvaWNlX2ZpX3ZveF9wb3B1bGlfZmlfMjAyNDEyMDYuc2FmZXRlbnNvcnMiLCAidm9jYWJfbmFtZSI6ICJ2b2NhYi50eHQifSwKICAgICLwn4ex8J+HuyBUaeG6v25nIExhdHZpYSAoUmFpdmlzRGVqdXMvRjUtVFRTLUxhdHZpYW4pIjogeyJtb2RlbCI6ICJGNVRUU19CYXNlIiwgInJlcG9faWQiOiAiUmFpdmlzRGVqdXMvRjUtVFRTLUxhdHZpYW4iLCAic3ViZm9sZGVyIjogTm9uZSwgImNrcHRfbmFtZSI6ICJtb2RlbC5zYWZldGVuc29ycyIsICJ2b2NhYl9uYW1lIjogInZvY2FiLnR4dCJ9LAogICAgIvCfh67wn4ezIFRp4bq/bmcgSGluZGkgKFNQUklOR0xhYi9GNS1IaW5kaS0yNEtIeikiOiB7Im1vZGVsIjogIkY1VFRTX3NtYWxsIiwgInJlcG9faWQiOiAiU1BSSU5HTGFiL0Y1LUhpbmRpLTI0S0h6IiwgInN1YmZvbGRlciI6IE5vbmUsICJja3B0X25hbWUiOiAibW9kZWxfMjUwMDAwMC5zYWZldGVuc29ycyIsICJ2b2NhYl9uYW1lIjogInZvY2FiLnR4dCJ9LAp9CgpFTkdJTkVTID0ge30gICMgY2FjaGUgY8OhYyBlbmdpbmUgxJHDoyBu4bqhcCAoZMO5bmcgY2h1bmcgY2hvIGPDoWMgdGFzayBzYXUpCgpkZWYgbG9hZF9lbmdpbmUobW9kZWxfa2V5KToKICAgICIiIk7huqFwIChob+G6t2MgbOG6pXkgbOG6oWkgbuG6v3UgxJHDoyBu4bqhcCkgZW5naW5lIGNobyBt4buZdCBtb2RlbF9rZXkgdHJvbmcgTU9ERUxTLiIiIgogICAgaWYgbW9kZWxfa2V5IGluIEVOR0lORVM6CiAgICAgICAgcmV0dXJuIEVOR0lORVNbbW9kZWxfa2V5XQogICAgaW5mbyA9IE1PREVMU1ttb2RlbF9rZXldCiAgICBwcmludChmIsSQYW5nIHThuqNpIHbDoCBu4bqhcCBtb2RlbDoge21vZGVsX2tleX0gLi4uIikKICAgIGlmIGluZm9bInJlcG9faWQiXToKICAgICAgICBja3B0ID0gaGZfaHViX2Rvd25sb2FkKGluZm9bInJlcG9faWQiXSwgaW5mb1siY2twdF9uYW1lIl0sIHN1YmZvbGRlcj1pbmZvWyJzdWJmb2xkZXIiXSkKICAgICAgICB2b2NhYiA9IGhmX2h1Yl9kb3dubG9hZChpbmZvWyJyZXBvX2lkIl0sIGluZm9bInZvY2FiX25hbWUiXSwgc3ViZm9sZGVyPWluZm9bInN1YmZvbGRlciJdKQogICAgICAgIEVOR0lORVNbbW9kZWxfa2V5XSA9IEY1VFRTKG1vZGVsPWluZm9bIm1vZGVsIl0sIGNrcHRfZmlsZT1ja3B0LCB2b2NhYl9maWxlPXZvY2FiKQogICAgZWxzZToKICAgICAgICBFTkdJTkVTW21vZGVsX2tleV0gPSBGNVRUUyhtb2RlbD1pbmZvWyJtb2RlbCJdKQogICAgcmV0dXJuIEVOR0lORVNbbW9kZWxfa2V5XQoKVklFVE5BTUVTRV9NT0RFTCA9IEZhbHNlICAjIFRydWUgPSBu4bqhcCBz4bq1biBtb2RlbCB0aeG6v25nIFZp4buHdCBuZ2F5IHThu6sgxJHhuqd1CkRFRkFVTFRfTU9ERUwgPSAi8J+Hu/Cfh7MgVGnhur9uZyBWaeG7h3QgKEY1LVRUUy1WaWV0bmFtZXNlKSIgaWYgVklFVE5BTUVTRV9NT0RFTCBlbHNlICLwn4es8J+Hp/Cfh6jwn4ezIEY1LVRUUyB2MSBCYXNlIChBbmggKyBUcnVuZykiCmVuZ2luZSA9IGxvYWRfZW5naW5lKERFRkFVTFRfTU9ERUwpCnByaW50KCJPSyDigJQgbW9kZWwgxJHDoyBz4bq1biBzw6BuZzoiLCBERUZBVUxUX01PREVMKQo=").decode("utf-8"))

In [ ]:
# @title TASK 4 — HÀM SINH ỔN ĐỊNH (chống thiếu chữ / vấp)import base64 as _b64exec(_b64.b64decode("IyBAdGl0bGUgVEFTSyA0IOKAlCBIw4BNIFNJTkgg4buUTiDEkOG7ik5IIChjaOG7kW5nIHRoaeG6v3UgY2jhu68gLyB24bqlcCkKIyBMw70gZG8gZ+G7kWMgY+G7p2EgbOG7l2kgInRoaeG6v3UgY2jhu68iIHbDoCAiduG6pXAiOiBGNS1UVFMgY2hpYSB2xINuIGLhuqNuIHRow6BuaCBjw6FjIMSRb+G6oW4KIyAoY2h1bmspIGPDsyDEkeG7mSBkw6BpIHThu7EgxJHhu5luZyBjw7MgdGjhu4MgcuG6pXQgbOG7m24sIHbDoCDEkW/huqFuIGN14buRaSBt4buXaSBjaHVuayBoYXkgYuG7iyBj4bqvdCBt4bqldCB04burLgojIEjDoG0gZMaw4bubaSDEkcOieSDDqXAgY2hpYSB0aGVvIEPDglUgduG7m2kgxJHhu5kgZMOgaSBD4buQIMSQ4buKTkgsIHbDoCB0aMOqbSBk4bqldSBjw6J1IMSR4bqneSDEkeG7pyBjaG8gbeG7l2kgxJFv4bqhbi4KaW1wb3J0IG9zLCByYW5kb20sIHN5cwppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaGF1ZGlvCmltcG9ydCBzb3VuZGZpbGUgYXMgc2YKZnJvbSBmNV90dHMuaW5mZXIudXRpbHNfaW5mZXIgaW1wb3J0ICgKICAgIHByZXByb2Nlc3NfcmVmX2F1ZGlvX3RleHQsCiAgICBjaHVua190ZXh0LAogICAgaW5mZXJfYmF0Y2hfcHJvY2VzcywKICAgIHRhcmdldF9zYW1wbGVfcmF0ZSwKICAgIHRhcmdldF9ybXMsCiAgICBjcm9zc19mYWRlX2R1cmF0aW9uLAogICAgcmVtb3ZlX3NpbGVuY2VfZm9yX2dlbmVyYXRlZF93YXYsCikKZnJvbSBmNV90dHMubW9kZWwudXRpbHMgaW1wb3J0IHNlZWRfZXZlcnl0aGluZwoKCmRlZiBfc3BsaXRfY2xlYW4odGV4dCwgbWF4X2NoYXJzKToKICAgICIiIkNoaWEgdsSDbiBi4bqjbiB0aGVvIGPDonUsIGdp4bubaSBo4bqhbiBt4buXaSDEkW/huqFuIH5tYXhfY2hhcnMga8O9IHThu7EsIGx1w7RuIGvhur90IHRow7pjIGLhurFuZyBk4bqldSBjw6J1LiIiIgogICAgYmF0Y2hlcyA9IGNodW5rX3RleHQodGV4dCwgbWF4X2NoYXJzPWludChtYXhfY2hhcnMpKQogICAgb3V0ID0gW10KICAgIGZvciBiIGluIGJhdGNoZXM6CiAgICAgICAgYiA9IGIuc3RyaXAoKQogICAgICAgIGlmIG5vdCBiOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmIGJbLTFdIG5vdCBpbiAiLiE/44CC77yB77yfIjoKICAgICAgICAgICAgYiA9IGIgKyAiLiIKICAgICAgICBvdXQuYXBwZW5kKGIpCiAgICByZXR1cm4gb3V0CgoKZGVmIHByZXBhcmVfcmVmKHJlZl9maWxlLCByZWZfdGV4dCk6CiAgICAiIiJDaOG7kW5nIGzhu5dpIFwibsOzaSB0aOG7q2EgLyBs4bq3cCBs4bqhaSAxIMSRb+G6oW4gdsSDbiBi4bqjblwiOiBjaOG7iSBLSeG7gk0gVFJBLCBraMO0bmcgc+G7rWEgZmlsZS4KICAgIE7hur91IGdp4buNbmcgbeG6q3UgZMOgaSBoxqFuIDEycywgRjUtVFRTIHPhur0gdOG7sSBj4bqvdCBj4buldCBhdWRpbyBsw7pjIHByZXByb2Nlc3MgbmjGsG5nIGdp4buvIG5ndXnDqm4KICAgIHJlZl90ZXh0IC0+IGF1ZGlvIGzhu4djaCB0ZXh0IC0+IG1vZGVsIMSR4buNYyB0aMOqbS9s4bq3cCBs4bqhaSBj4bulbSBjdeG7kWkgY+G7p2EgcmVmX3RleHQuCiAgICDhu54gxJHDonkgY2jhu4kgaW4gY+G6o25oIGLDoW8gxJHhu4MgYuG6oW4gdOG7sSB44butIGzDvSAoYW4gdG/DoG4sIGtow7RuZyDEkeG7lWkgaMOgbmggdmkgc28gduG7m2kgYuG6o24gZ+G7kWMpLiIiIgogICAgdHJ5OgogICAgICAgIHdhdiwgc3IgPSB0b3JjaGF1ZGlvLmxvYWQocmVmX2ZpbGUpCiAgICAgICAgZHVyYXRpb24gPSB3YXYuc2hhcGVbMV0gLyBzcgogICAgICAgIGlmIGR1cmF0aW9uID4gMTIuMDoKICAgICAgICAgICAgcHJpbnQoZiLimqDvuI8gR2nhu41uZyBt4bqrdSBkw6BpIHtkdXJhdGlvbjouMWZ9cyAoPiAxMnMpOiBGNS1UVFMgc+G6vSBj4bqvdCBj4buldCBnaeG7jW5nIG3huqt1LiAiCiAgICAgICAgICAgICAgICAgIGYiTsOqbiBkw7luZyBnaeG7jW5nIG3huqt1IDwgMTJzIGhv4bq3YyDEkeG7gyB0cuG7kW5nICdO4buZaSBkdW5nIHRyb25nIGZpbGUgZ2nhu41uZyBt4bqrdScgIgogICAgICAgICAgICAgICAgICBmIsSR4buDIHRyw6FuaCBuw7NpIHRo4burYS9s4bq3cCAxIMSRb+G6oW4uIikKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcGFzcwogICAgcmV0dXJuIHJlZl9maWxlLCByZWZfdGV4dCBvciAiIgoKCmRlZiBzdGFibGVfZ2VuZXJhdGUoZW5naW5lLCByZWZfZmlsZSwgcmVmX3RleHQsIGdlbl90ZXh0LCBtYXhfY2hhcnM9MTAwLAogICAgICAgICAgICAgICAgICAgIHNwZWVkPTEuMCwgbmZlPTMyLCBjZmc9Mi4wLCBzZWVkPU5vbmUpOgogICAgIiIiU2luaCBnaeG7jW5nIOG7lW4gxJHhu4tuaDogdHLhuqMgduG7gSAod2F2ZSwgc3IsIHNwZWMsIHNlZWRfZGFfZHVuZykuIiIiCiAgICBpZiBzZWVkIGlzIE5vbmU6CiAgICAgICAgc2VlZCA9IHJhbmRvbS5yYW5kaW50KDAsIHN5cy5tYXhzaXplKQogICAgc2VlZF9ldmVyeXRoaW5nKHNlZWQpCgogICAgcmVmX2F1ZGlvLCByZWZfdGV4dCA9IHByZXBhcmVfcmVmKHJlZl9maWxlLCByZWZfdGV4dCBvciAiIikKICAgIHJlZl9hdWRpbywgcmVmX3RleHQgPSBwcmVwcm9jZXNzX3JlZl9hdWRpb190ZXh0KHJlZl9hdWRpbywgcmVmX3RleHQsIHNob3dfaW5mbz1wcmludCkKICAgIGJhdGNoZXMgPSBfc3BsaXRfY2xlYW4oZ2VuX3RleHQsIG1heF9jaGFycz1tYXhfY2hhcnMpCiAgICBwcmludCgiU+G7kSDEkW/huqFuIGPhuqduIHNpbmg6IiwgbGVuKGJhdGNoZXMpKQogICAgaWYgbm90IGJhdGNoZXM6CiAgICAgICAgcmV0dXJuIE5vbmUsIHRhcmdldF9zYW1wbGVfcmF0ZSwgTm9uZSwgc2VlZAoKICAgIGF1ZGlvLCBzciA9IHRvcmNoYXVkaW8ubG9hZChyZWZfYXVkaW8pCiAgICBnZW4gPSBpbmZlcl9iYXRjaF9wcm9jZXNzKAogICAgICAgIChhdWRpbywgc3IpLAogICAgICAgIHJlZl90ZXh0LAogICAgICAgIGJhdGNoZXMsCiAgICAgICAgZW5naW5lLmVtYV9tb2RlbCwKICAgICAgICBlbmdpbmUudm9jb2RlciwKICAgICAgICBtZWxfc3BlY190eXBlPWVuZ2luZS5tZWxfc3BlY190eXBlLAogICAgICAgIHByb2dyZXNzPU5vbmUsCiAgICAgICAgdGFyZ2V0X3Jtcz10YXJnZXRfcm1zLAogICAgICAgIGNyb3NzX2ZhZGVfZHVyYXRpb249Y3Jvc3NfZmFkZV9kdXJhdGlvbiwKICAgICAgICBuZmVfc3RlcD1pbnQobmZlKSwKICAgICAgICBjZmdfc3RyZW5ndGg9ZmxvYXQoY2ZnKSwKICAgICAgICBzd2F5X3NhbXBsaW5nX2NvZWY9LTEuMCwKICAgICAgICBzcGVlZD1zcGVlZCwKICAgICAgICBmaXhfZHVyYXRpb249Tm9uZSwKICAgICAgICBkZXZpY2U9ZW5naW5lLmRldmljZSwKICAgICkKICAgIHRyeToKICAgICAgICB3YXZlLCBzLCBzcGVjID0gbmV4dChnZW4pCiAgICBleGNlcHQgU3RvcEl0ZXJhdGlvbjoKICAgICAgICB3YXZlLCBzLCBzcGVjID0gTm9uZSwgdGFyZ2V0X3NhbXBsZV9yYXRlLCBOb25lCiAgICByZXR1cm4gd2F2ZSwgcywgc3BlYywgc2VlZAoKCmRlZiBzYXZlX3dhdih3YXZlLCBzciwgcGF0aCwgcmVtb3ZlX3NpbGVuY2U9RmFsc2UpOgogICAgc2Yud3JpdGUocGF0aCwgd2F2ZSwgc3IpCiAgICBpZiByZW1vdmVfc2lsZW5jZToKICAgICAgICByZW1vdmVfc2lsZW5jZV9mb3JfZ2VuZXJhdGVkX3dhdihwYXRoKQogICAgcmV0dXJuIHBhdGgKCmRlZiB0b19tcDMocGF0aCwgYml0cmF0ZT0xMjgpOgogICAgIiIiTsOpbiBXQVYgLT4gTVAzIMSR4buDIGZpbGUgbmjhu48sIHBow6F0L3R1YSBtxrDhu6N0IHRyw6puIENvbGFiLgogICAgKFdBViBxdcOhIHRvIG7Dqm4gdHLDrG5oIHBow6F0IGNo4buJIHR1YSDEkcaw4bujYyBwaOG6p24gxJHDoyB04bqjaSB24buBOyB0dWEgeGEgYuG7iyBuaOG6o3kgduG7gSBjaOG7lyDEkWFuZyBwaMOhdC4pIiIiCiAgICBtcDNfcGF0aCA9IG9zLnBhdGguc3BsaXRleHQocGF0aClbMF0gKyAiLm1wMyIKICAgIHRyeToKICAgICAgICBvcy5zeXN0ZW0oZidmZm1wZWcgLXkgLWxvZ2xldmVsIGVycm9yIC1pICJ7cGF0aH0iIC1iOmEge2JpdHJhdGV9ayAtd3JpdGVfeGluZyAxICJ7bXAzX3BhdGh9IicpCiAgICAgICAgaWYgb3MucGF0aC5leGlzdHMobXAzX3BhdGgpIGFuZCBvcy5wYXRoLmdldHNpemUobXAzX3BhdGgpID4gMDoKICAgICAgICAgICAgcmV0dXJuIG1wM19wYXRoCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHBhc3MKICAgIHJldHVybiBwYXRoCg==").decode("utf-8"))

In [ ]:
# @title TASK 5 — CHẠY THỬ (dùng giọng mẫu có sẵn)import base64 as _b64exec(_b64.b64decode("IyBAdGl0bGUgVEFTSyA1IOKAlCBDSOG6oFkgVEjhu6wgKGTDuW5nIGdp4buNbmcgbeG6q3UgY8OzIHPhurVuKQppbXBvcnQgb3MKZnJvbSBpbXBvcnRsaWIucmVzb3VyY2VzIGltcG9ydCBmaWxlcwoKb3MubWFrZWRpcnMoIi9jb250ZW50L291dHB1dCIsIGV4aXN0X29rPVRydWUpCgpyZWZfYXVkaW8gPSBzdHIoZmlsZXMoImY1X3R0cyIpLmpvaW5wYXRoKCJpbmZlci9leGFtcGxlcy9iYXNpYy9iYXNpY19yZWZfZW4ud2F2IikpCnJlZl90ZXh0ID0gIlNvbWUgY2FsbCBtZSBuYXR1cmUsIG90aGVycyBjYWxsIG1lIG1vdGhlciBuYXR1cmUuIgpnZW5fdGV4dCA9ICJIZWxsbyEgVGhpcyBpcyBhIHF1aWNrIHRlc3Qgb2YgeW91ciBjbG9uZWQgdm9pY2UuIEtlZXAgdGhpcyBmaWxlIGFzIHlvdXIgcmVmZXJlbmNlIHZvaWNlIHNhbXBsZS4gIiBcCiAgICAgICAgICAgIlRoZSB0ZXh0IGlzIHNwbGl0IGludG8gc21hbGwgY29tcGxldGUgc2VudGVuY2VzIHNvIHRoYXQgZXZlcnkgc2luZ2xlIHdvcmQgaXMgcHJvbm91bmNlZCBmdWxseSBhbmQgY2xlYXJseS4iCgp3YXZlLCBzciwgc3BlYywgc2VlZCA9IHN0YWJsZV9nZW5lcmF0ZShlbmdpbmUsIHJlZl9hdWRpbywgcmVmX3RleHQsIGdlbl90ZXh0LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtYXhfY2hhcnM9MTAwLCBzcGVlZD0xLjAsIG5mZT0zMiwgY2ZnPTIuMCkKb3V0X3BhdGggPSBzYXZlX3dhdih3YXZlLCBzciwgIi9jb250ZW50L291dHB1dC90ZXN0X291dHB1dC53YXYiKQpwcmludCgixJDDoyB04bqhbzoiLCBvdXRfcGF0aCwgInwgc2VlZCA9Iiwgc2VlZCwgInwgc3IgPSIsIHNyKQoKZnJvbSBJUHl0aG9uLmRpc3BsYXkgaW1wb3J0IEF1ZGlvCkF1ZGlvKHRvX21wMyhvdXRfcGF0aCkpCg==").decode("utf-8"))

In [ ]:
# @title TASK 6 — GẮN GOOGLE DRIVE (tùy chọn, để giọng lưu được lâu dài)import base64 as _b64exec(_b64.b64decode("IyBAdGl0bGUgVEFTSyA2IOKAlCBH4bquTiBHT09HTEUgRFJJVkUgKHTDuXkgY2jhu41uLCDEkeG7gyBnaeG7jW5nIGzGsHUgxJHGsOG7o2MgbMOidSBkw6BpKQojIEtIw5RORyBi4bqvdCBideG7mWMuIENo4bqheSBjZWxsIG7DoHkgbuG6v3UgbXXhu5FuIGPDoWMgZ2nhu41uZyDEkcOjIGzGsHUgxJHGsOG7o2MgZ2nhu68gbOG6oWkgZ2nhu69hIGPDoWMgcGhpw6puIENvbGFiLgojIChLaMO0bmcgZ+G6r24gRHJpdmUgdGjDrCBnaeG7jW5nIHbhuqtuIGzGsHUgxJHGsOG7o2MgdHJvbmcgcGhpw6puIGhp4buHbiB04bqhaSwgbmjGsG5nIG3huqV0IGtoaSDEkcOzbmcgcnVudGltZS4pCnRyeToKICAgIGZyb20gZ29vZ2xlLmNvbGFiIGltcG9ydCBkcml2ZQogICAgZHJpdmUubW91bnQoIi9jb250ZW50L2RyaXZlIikKICAgIHByaW50KCLEkMOjIGfhuq9uIEdvb2dsZSBEcml2ZS4gR2nhu41uZyDEkcOjIGzGsHUgc+G6vSDEkcaw4bujYyBsxrB1IHbDoG8gTXlEcml2ZS9WT0lDRV9DTE9ORV9UVFMvdm9pY2VzLyIpCmV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgIHByaW50KCJC4buPIHF1YSBn4bqvbiBEcml2ZSAoa2jDtG5nIHNhbyk6IiwgZSkK").decode("utf-8"))

In [ ]:
# @title TASK 7 — THƯ VIỆN GIỌNG ĐÃ LƯU (lưu / nạp lại giọng mẫu đã dùng)import base64 as _b64exec(_b64.b64decode("IyBAdGl0bGUgVEFTSyA3IOKAlCBUSMavIFZJ4buGTiBHSeG7jE5HIMSQw4MgTMavVSAobMawdSAvIG7huqFwIGzhuqFpIGdp4buNbmcgbeG6q3UgxJHDoyBkw7luZykKIyBMxrB1IG5oaeG7gXUgZ2nhu41uZyBt4bqrdSDEkcOjIGTDuW5nIChrw6htIG7hu5lpIGR1bmcpIMSR4buDIGTDuW5nIGzhuqFpIG5oYW5oLiBO4bq/dSDEkcOjIGfhuq9uIEdvb2dsZSBEcml2ZQojIChjZWxsIFRBU0sgNikgZ2nhu41uZyBz4bq9IMSRxrDhu6NjIGdp4buvIGzDonUgZMOgaTsgbuG6v3Uga2jDtG5nLCBnaeG7jW5nIGzGsHUgdHJvbmcgcGhpw6puIGhp4buHbiB04bqhaS4KaW1wb3J0IG9zLCBqc29uLCBzaHV0aWwsIHJlCgpWT0lDRV9ESVIgPSAiL2NvbnRlbnQvdm9pY2VzIgpEUklWRV9WT0lDRV9ESVIgPSAiL2NvbnRlbnQvZHJpdmUvTXlEcml2ZS9WT0lDRV9DTE9ORV9UVFMvdm9pY2VzIgoKZGVmIF92b2ljZXNfcm9vdCgpOgogICAgaWYgb3MucGF0aC5pc2RpcigiL2NvbnRlbnQvZHJpdmUvTXlEcml2ZSIpOgogICAgICAgIHJldHVybiBEUklWRV9WT0lDRV9ESVIKICAgIHJldHVybiBWT0lDRV9ESVIKCmRlZiBfbG9hZF9pbmRleCgpOgogICAgaW5kZXhfcGF0aCA9IG9zLnBhdGguam9pbihfdm9pY2VzX3Jvb3QoKSwgInZvaWNlcy5qc29uIikKICAgIGlmIG9zLnBhdGguZXhpc3RzKGluZGV4X3BhdGgpOgogICAgICAgIHdpdGggb3BlbihpbmRleF9wYXRoLCAiciIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgICAgIHJldHVybiBqc29uLmxvYWQoZikKICAgIHJldHVybiB7fQoKZGVmIF9zYXZlX2luZGV4KGluZGV4KToKICAgIHJvb3QgPSBfdm9pY2VzX3Jvb3QoKQogICAgb3MubWFrZWRpcnMocm9vdCwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHdpdGggb3Blbihvcy5wYXRoLmpvaW4ocm9vdCwgInZvaWNlcy5qc29uIiksICJ3IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICBqc29uLmR1bXAoaW5kZXgsIGYsIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIpCgpkZWYgX3NhZmVfbmFtZShuYW1lKToKICAgIG5hbWUgPSByZS5zdWIociJbXlx3XC0gXSIsICIiLCBuYW1lKS5zdHJpcCgpCiAgICByZXR1cm4gbmFtZSBvciAidm9pY2UiCgpBVURJT19FWFRTID0gKCIud2F2IiwgIi5tcDMiLCAiLmZsYWMiLCAiLm00YSIsICIub2dnIiwgIi53bWEiKQoKZGVmIF9tb2RlbF9mb2xkZXIobW9kZWwpOgogICAgIiIiVMOqbiB0aMawIG3hu6VjIHJpw6puZyBjaG8gbeG7l2kgbmfDtG4gbmfhu68vbW9kZWwgKG3hu5dpIG1vZGVsID0gMSB0aMawIG3hu6VjIGdp4buNbmcpLiIiIgogICAgaWYgbm90IG1vZGVsOgogICAgICAgIHJldHVybiAiQ2h1bmciCiAgICBmID0gcmUuc3ViKHIiW15cd10rIiwgIl8iLCBtb2RlbCkuc3RyaXAoIl8iKQogICAgZiA9IHJlLnN1YihyIl8rIiwgIl8iLCBmKQogICAgcmV0dXJuIGYgb3IgIkNodW5nIgoKZGVmIF9tb2RlbF9kaXIobW9kZWwpOgogICAgcmV0dXJuIG9zLnBhdGguam9pbihfdm9pY2VzX3Jvb3QoKSwgX21vZGVsX2ZvbGRlcihtb2RlbCkpCgpkZWYgX21vZGVsX2Zyb21fZm9sZGVyKGZvbGRlcik6CiAgICAiIiJOZ8aw4bujYyBs4bqhaTogdMOqbiB0aMawIG3hu6VjIC0+IG1vZGVsIGtleS4iIiIKICAgIGlmICJNT0RFTFMiIGluIGdsb2JhbHMoKToKICAgICAgICBmb3IgbWsgaW4gTU9ERUxTOgogICAgICAgICAgICBpZiBfbW9kZWxfZm9sZGVyKG1rKSA9PSBmb2xkZXI6CiAgICAgICAgICAgICAgICByZXR1cm4gbWsKICAgIHJldHVybiBmb2xkZXIKCmRlZiBsaXN0X3ZvaWNlcygpOgogICAgcmV0dXJuIGxpc3Rfdm9pY2VzX2J5X21vZGVsKE5vbmUpCgpkZWYgbGlzdF92b2ljZXNfYnlfbW9kZWwobW9kZWw9Tm9uZSk6CiAgICAiIiJMaeG7h3Qga8OqIGdp4buNbmcgY+G7p2EgMSBuZ8O0biBuZ+G7ryBi4bqxbmcgY8OhY2ggUVXDiVQgdGjGsCBt4bulYyBtb2RlbCDEkcOzLgogICAgQuG6oW4gY8OzIHRo4buDIGLhu48gdGjhurNuZyBmaWxlIFdBVi9NUDMgdsOgbyB0aMawIG3hu6VjIGdp4buNbmcgY+G7p2EgbmfDtG4gbmfhu68gbMOgIG7DsyB04buxIGhp4buHbiByYS4iIiIKICAgIGluZGV4ID0gX2xvYWRfaW5kZXgoKQogICAgbmFtZXMgPSBzZXQoKQogICAgaWYgbW9kZWwgaXMgbm90IE5vbmU6CiAgICAgICAgZm9sZGVyID0gX21vZGVsX2Rpcihtb2RlbCkKICAgICAgICBpZiBvcy5wYXRoLmlzZGlyKGZvbGRlcik6CiAgICAgICAgICAgIGZvciBmbiBpbiBvcy5saXN0ZGlyKGZvbGRlcik6CiAgICAgICAgICAgICAgICBpZiBmbi5sb3dlcigpLmVuZHN3aXRoKEFVRElPX0VYVFMpOgogICAgICAgICAgICAgICAgICAgIG5hbWVzLmFkZChmbikKICAgICAgICBmb3IgdiBpbiBpbmRleC52YWx1ZXMoKToKICAgICAgICAgICAgaWYgKG5vdCB2LmdldCgibW9kZWwiKSBvciB2WyJtb2RlbCJdID09IG1vZGVsKSBhbmQgdi5nZXQoImF1ZGlvIik6CiAgICAgICAgICAgICAgICBuYW1lcy5hZGQob3MucGF0aC5iYXNlbmFtZSh2WyJhdWRpbyJdKSkKICAgIGVsc2U6CiAgICAgICAgZm9yIF8sIF8sIGZpbGVzIGluIG9zLndhbGsoX3ZvaWNlc19yb290KCkpOgogICAgICAgICAgICBmb3IgZm4gaW4gZmlsZXM6CiAgICAgICAgICAgICAgICBpZiBmbi5sb3dlcigpLmVuZHN3aXRoKEFVRElPX0VYVFMpOgogICAgICAgICAgICAgICAgICAgIG5hbWVzLmFkZChmbikKICAgICAgICBmb3IgdiBpbiBpbmRleC52YWx1ZXMoKToKICAgICAgICAgICAgaWYgdi5nZXQoImF1ZGlvIik6CiAgICAgICAgICAgICAgICBuYW1lcy5hZGQob3MucGF0aC5iYXNlbmFtZSh2WyJhdWRpbyJdKSkKICAgIHJldHVybiBzb3J0ZWQobmFtZXMsIGtleT1zdHIubG93ZXIpCgpkZWYgc2F2ZV92b2ljZShuYW1lLCBhdWRpb19wYXRoLCByZWZfdGV4dCwgbW9kZWw9Tm9uZSk6CiAgICAiIiJMxrB1IGdp4buNbmcgdsOgbyBUSMavIE3hu6RDIGPhu6dhIG1vZGVsIChuZ8O0biBuZ+G7rykgxJFhbmcgY2jhu41uLiIiIgogICAgaWYgbm90IChuYW1lIG9yICIiKS5zdHJpcCgpOgogICAgICAgIHJldHVybiBsaXN0X3ZvaWNlc19ieV9tb2RlbChtb2RlbCksICLimqDvuI8gQ2jGsGEgbmjhuq1wIFTDqm4gZ2nhu41uZyDEkeG7gyBsxrB1LiIKICAgIGlmIG5vdCBhdWRpb19wYXRoOgogICAgICAgIHJldHVybiBsaXN0X3ZvaWNlc19ieV9tb2RlbChtb2RlbCksICLimqDvuI8gQ2jGsGEgY8OzIGZpbGUgZ2nhu41uZyBt4bqrdSAocmVmIGF1ZGlvKSDEkeG7gyBsxrB1LiIKICAgIG5hbWUgPSBfc2FmZV9uYW1lKG5hbWUpCiAgICBleHQgPSBvcy5wYXRoLnNwbGl0ZXh0KGF1ZGlvX3BhdGgpWzFdIG9yICIud2F2IgogICAgZm9sZGVyID0gX21vZGVsX2ZvbGRlcihtb2RlbCkKICAgIGRlc3RfZGlyID0gb3MucGF0aC5qb2luKF92b2ljZXNfcm9vdCgpLCBmb2xkZXIpCiAgICBvcy5tYWtlZGlycyhkZXN0X2RpciwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGRlc3QgPSBvcy5wYXRoLmpvaW4oZGVzdF9kaXIsIG5hbWUgKyBleHQpCiAgICBmb3IgZm4gaW4gb3MubGlzdGRpcihkZXN0X2Rpcik6ICAjIHhvYSBjYWMgZmlsZSBjdW5nIHRlbiBraGFjIGR1b2kgKC53YXYvLm1wMykgZGUga2hvbmcgdHJ1bmcKICAgICAgICBpZiBmbi5sb3dlcigpLmVuZHN3aXRoKEFVRElPX0VYVFMpIGFuZCBvcy5wYXRoLnNwbGl0ZXh0KGZuKVswXSA9PSBuYW1lIGFuZCBvcy5wYXRoLmFic3BhdGgob3MucGF0aC5qb2luKGRlc3RfZGlyLCBmbikpICE9IG9zLnBhdGguYWJzcGF0aChkZXN0KToKICAgICAgICAgICAgdHJ5OiBvcy5yZW1vdmUob3MucGF0aC5qb2luKGRlc3RfZGlyLCBmbikpCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246IHBhc3MKICAgIHNodXRpbC5jb3B5ZmlsZShhdWRpb19wYXRoLCBkZXN0KQogICAgaW5kZXggPSBfbG9hZF9pbmRleCgpCiAgICBpbmRleFtuYW1lXSA9IHsiYXVkaW8iOiBkZXN0LCAicmVmX3RleHQiOiByZWZfdGV4dCBvciAiIiwgIm1vZGVsIjogbW9kZWwgb3IgIiJ9CiAgICBfc2F2ZV9pbmRleChpbmRleCkKICAgIHJldHVybiBsaXN0X3ZvaWNlc19ieV9tb2RlbChtb2RlbCksIGYixJDDoyBsxrB1IGdp4buNbmcge25hbWV9IHbDoG8gdGjGsCBt4bulYyB7Zm9sZGVyfS8g4oCUIGNo4buNbiBs4bqhaSBsw6AgZMO5bmcgxJHGsOG7o2MuIgoKZGVmIGxvYWRfdm9pY2UobmFtZSk6CiAgICAiIiJO4bqhcCBnaeG7jW5nIC0+IChwYXRoLCByZWZfdGV4dCwgbW9kZWwpLiBI4buXIHRy4bujIHTDqm4gY8OzL2tob+G6o25nIMSRdcO0aSAud2F2Ly5tcDMuIiIiCiAgICBpZiBub3QgbmFtZToKICAgICAgICByZXR1cm4gTm9uZSwgTm9uZSwgIiIKICAgIGluZGV4ID0gX2xvYWRfaW5kZXgoKQogICAgdiA9IGluZGV4LmdldChuYW1lKQogICAgaWYgdiBpcyBOb25lOgogICAgICAgIGZvciBrLCBpdCBpbiBpbmRleC5pdGVtcygpOgogICAgICAgICAgICBpZiBvcy5wYXRoLmJhc2VuYW1lKGl0WyJhdWRpbyJdKSA9PSBuYW1lOgogICAgICAgICAgICAgICAgdiA9IGl0CiAgICAgICAgICAgICAgICBicmVhawogICAgaWYgdiBhbmQgdi5nZXQoImF1ZGlvIikgYW5kIG9zLnBhdGguZXhpc3RzKHZbImF1ZGlvIl0pOgogICAgICAgIHJldHVybiB2WyJhdWRpbyJdLCB2LmdldCgicmVmX3RleHQiLCAiIiksIHYuZ2V0KCJtb2RlbCIsICIiKQogICAgcGF0aCwgcmVmX3RleHQsIG1vZGVsID0gTm9uZSwgIiIsICIiCiAgICBmb3Igcm9vdF8sIF8sIGZpbGVzIGluIG9zLndhbGsoX3ZvaWNlc19yb290KCkpOgogICAgICAgIGZvciBmbiBpbiBmaWxlczoKICAgICAgICAgICAgaWYgZm4ubG93ZXIoKS5lbmRzd2l0aChBVURJT19FWFRTKSBhbmQgKGZuID09IG5hbWUgb3Igb3MucGF0aC5zcGxpdGV4dChmbilbMF0gPT0gbmFtZSk6CiAgICAgICAgICAgICAgICBwYXRoID0gb3MucGF0aC5qb2luKHJvb3RfLCBmbikKICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgaWYgcGF0aDoKICAgICAgICAgICAgYnJlYWsKICAgIGlmIHBhdGg6CiAgICAgICAgZm9sZGVyID0gb3MucGF0aC5iYXNlbmFtZShvcy5wYXRoLmRpcm5hbWUocGF0aCkpCiAgICAgICAgbW9kZWwgPSBfbW9kZWxfZnJvbV9mb2xkZXIoZm9sZGVyKQogICAgICAgIGZvciBrLCBpdCBpbiBpbmRleC5pdGVtcygpOgogICAgICAgICAgICBpZiBvcy5wYXRoLmJhc2VuYW1lKGl0WyJhdWRpbyJdKSA9PSBvcy5wYXRoLmJhc2VuYW1lKHBhdGgpOgogICAgICAgICAgICAgICAgcmVmX3RleHQgPSBpdC5nZXQoInJlZl90ZXh0IiwgIiIpCiAgICAgICAgICAgICAgICBtb2RlbCA9IGl0LmdldCgibW9kZWwiLCAiIikgb3IgbW9kZWwKICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgcmV0dXJuIHBhdGgsIHJlZl90ZXh0LCBtb2RlbAogICAgcmV0dXJuIE5vbmUsIE5vbmUsICIiCgpkZWYgZGVsZXRlX3ZvaWNlKG5hbWUpOgogICAgIiIiWMOzYSBnaeG7jW5nICh0aGVvIGluZGV4IGhv4bq3YyBmaWxlIHRy4buxYyB0aeG6v3AgdHJvbmcgdGjGsCBt4bulYykuIiIiCiAgICBpbmRleCA9IF9sb2FkX2luZGV4KCkKICAgIGlmIG5hbWUgYW5kIG5hbWUgaW4gaW5kZXg6CiAgICAgICAgYXVkaW8gPSBpbmRleFtuYW1lXS5nZXQoImF1ZGlvIikKICAgICAgICBkZWwgaW5kZXhbbmFtZV0KICAgICAgICBfc2F2ZV9pbmRleChpbmRleCkKICAgICAgICBpZiBhdWRpbyBhbmQgb3MucGF0aC5leGlzdHMoYXVkaW8pOgogICAgICAgICAgICBvcy5yZW1vdmUoYXVkaW8pCiAgICAgICAgcmV0dXJuIGxpc3Rfdm9pY2VzKCksIGYixJDDoyB4w7NhIGdp4buNbmcge25hbWV9LiIKICAgIGZvciByb290XywgXywgZmlsZXMgaW4gb3Mud2Fsayhfdm9pY2VzX3Jvb3QoKSk6CiAgICAgICAgZm9yIGZuIGluIGZpbGVzOgogICAgICAgICAgICBpZiBmbiA9PSBuYW1lIG9yIG9zLnBhdGguc3BsaXRleHQoZm4pWzBdID09IG5hbWU6CiAgICAgICAgICAgICAgICBmcCA9IG9zLnBhdGguam9pbihyb290XywgZm4pCiAgICAgICAgICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyhmcCk6CiAgICAgICAgICAgICAgICAgICAgb3MucmVtb3ZlKGZwKQogICAgICAgICAgICAgICAgcmV0dXJuIGxpc3Rfdm9pY2VzKCksIGYixJDDoyB4w7NhIGdp4buNbmcge25hbWV9LiIKICAgIHJldHVybiBsaXN0X3ZvaWNlcygpLCAiS2jDtG5nIHTDrG0gdGjhuqV5IGdp4buNbmcgxJHDoyBjaOG7jW4uIgoKZGVmIF92b2ljZV9pc19zYXZlZChhdWRpbyk6CiAgICAiIiJLaeG7g20gdHJhIGZpbGUgYXVkaW8gxJHDoyDEkcaw4bujYyBsxrB1IHRyb25nIHRoxrAgdmnhu4duIGhheSBjaMawYS4iIiIKICAgIGlmIG5vdCBhdWRpbzoKICAgICAgICByZXR1cm4gRmFsc2UKICAgIGZvciB2IGluIF9sb2FkX2luZGV4KCkudmFsdWVzKCk6CiAgICAgICAgc2F2ZWQgPSB2LmdldCgiYXVkaW8iKQogICAgICAgIGlmIHNhdmVkIGFuZCBvcy5wYXRoLmFic3BhdGgoc2F2ZWQpID09IG9zLnBhdGguYWJzcGF0aChhdWRpbyk6CiAgICAgICAgICAgIHJldHVybiBUcnVlCiAgICByZXR1cm4gRmFsc2UKCmRlZiBfZW5zdXJlX3ZvaWNlX2RpcnMoKToKICAgICIiIlThu7EgdOG6oW8gdGjGsCBt4bulYyBnaeG7jW5nIGNobyB04burbmcgbmfDtG4gbmfhu68vbW9kZWwga2hpIGNo4bqheSBs4bqnbiDEkeG6p3UuIiIiCiAgICByb290ID0gX3ZvaWNlc19yb290KCkKICAgIG9zLm1ha2VkaXJzKHJvb3QsIGV4aXN0X29rPVRydWUpCiAgICBtYWRlID0gW10KICAgIGlmICJNT0RFTFMiIGluIGdsb2JhbHMoKToKICAgICAgICBmb3IgbWsgaW4gTU9ERUxTOgogICAgICAgICAgICBkID0gX21vZGVsX2RpcihtaykKICAgICAgICAgICAgb3MubWFrZWRpcnMoZCwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICAgICAgbWFkZS5hcHBlbmQoKF9tb2RlbF9mb2xkZXIobWspLCBkKSkKICAgIHJldHVybiByb290LCBtYWRlCgpyb290LCBtYWRlID0gX2Vuc3VyZV92b2ljZV9kaXJzKCkKcHJpbnQoIlRoxrAgbeG7pWMgY2jhu6lhIGdp4buNbmc6Iiwgcm9vdCkKcHJpbnQoIsSQw6MgdOG7sSB04bqhbyB0aMawIG3hu6VjIGdp4buNbmcgY2hvIHThu6tuZyBuZ8O0biBuZ+G7ryAobOG6p24gxJHhuqd1KToiKQpmb3IgZm9sZGVyLCBwYXRoIGluIG1hZGU6CiAgICBwcmludCgiICAiLCBmb2xkZXIsICItPiIsIHBhdGgpCnByaW50KCJT4buRIGdp4buNbmcgxJHDoyBsxrB1IGhp4buHbiBjw7M6IiwgbGVuKGxpc3Rfdm9pY2VzKCkpKQo=").decode("utf-8"))

In [ ]:
# @title TASK 8 — KHỞI ĐỘNG GIAO DIỆN NHÂN BẢN GIỌNG NÓIimport base64 as _b64exec(_b64.b64decode("...").decode("utf-8"))

In [ ]:
# @title DÙNG TRỰC TIẾP BẰNG HÀM (tùy chọn)import base64 as _b64exec(_b64.b64decode("IyBAdGl0bGUgRMOZTkcgVFLhu7BDIFRJ4bq+UCBC4bqwTkcgSMOATSAodMO5eSBjaOG7jW4pCmRlZiBjbG9uZV92b2ljZShyZWZfZmlsZSwgZ2VuX3RleHQsIHJlZl90ZXh0PSIiLCBvdXRwdXQ9Ii9jb250ZW50L291dHB1dC9yZXN1bHQud2F2IiwKICAgICAgICAgICAgICAgIHNwZWVkPTEuMCwgc2VlZD1Ob25lLCBtYXhfY2hhcnM9MTAwLCBuZmU9MzIsIGNmZz0yLjApOgogICAgIiIiTmjDom4gYuG6o24gZ2nhu41uZyDhu5VuIMSR4buLbmg6IHJlZl9maWxlID0gZmlsZSBnaeG7jW5nIG3huqt1LCBnZW5fdGV4dCA9IHbEg24gYuG6o24gY+G6p24gxJHhu41jLiIiIgogICAgZW5nID0gbG9hZF9lbmdpbmUoREVGQVVMVF9NT0RFTCkKICAgIHdhdmUsIHNyLCBzcGVjLCBzZWVkX3VzZWQgPSBzdGFibGVfZ2VuZXJhdGUoZW5nLCByZWZfZmlsZSwgcmVmX3RleHQsIGdlbl90ZXh0LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtYXhfY2hhcnM9bWF4X2NoYXJzLCBzcGVlZD1zcGVlZCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbmZlPW5mZSwgY2ZnPWNmZywgc2VlZD1zZWVkKQogICAgcGF0aCA9IHNhdmVfd2F2KHdhdmUsIHNyLCBvdXRwdXQsIHJlbW92ZV9zaWxlbmNlPVRydWUpCiAgICByZXR1cm4gcGF0aCwgc3IsIHNlZWRfdXNlZAoKIyBWw60gZOG7pSBjw6FjaCBkw7luZyAoYuG7jyBjb21tZW50IMSR4buDIGNo4bqheSk6CiMgb3V0LCBzciwgc2VlZF91c2VkID0gY2xvbmVfdm9pY2UoIi9jb250ZW50L215X3ZvaWNlLndhdiIsICJYaW4gY2jDoG8sIMSRw6J5IGzDoCBnaeG7jW5nIG7Ds2kgxJHGsOG7o2MgbmjDom4gYuG6o24uIikKIyBmcm9tIElQeXRob24uZGlzcGxheSBpbXBvcnQgQXVkaW8KIyBBdWRpbyhvdXQpCg==").decode("utf-8"))

## CÁCH TẢI KẾT QUẢ VỀ MÁY
* Trong giao diện, bấm biểu tượng tải xuống (download) của ô audio kết quả.
* Hoặc vào thư mục `/content/output/` trong Files (panel bên trái) và tải xuống.

## MẸO CHẤT LƯỢNG
* File giọng mẫu nên dài **5–12 giây**, rõ ràng, không nhạc nền, không ồn.
* **Nhập chính xác "Nội dung trong file giọng mẫu"** — nếu để trống máy phải tự nhận dạng bằng Whisper (chậm hơn và có thể sai, gây thiếu chữ).
* Bị **thiếu chữ**: giảm "Độ dài đoạn (max_chars)".
* Bị **vấp / lắp**: tăng nfe_step lên 48–64, hoặc tăng nhẹ max_chars lên 120–150.
* Với văn bản rất dài, máy tự chia nhỏ và ghép nối — cứ nhập thoải mái.

## GIỌNG ĐÃ LƯU
* Giọng đã lưu nằm trong thư mục `/content/voices/` (hoặc `MyDrive/VOICE_CLONE_TTS/voices/` nếu đã gắn Drive) — tải thư mục về máy để sao lưu.
